# Практикум 4. Множественная регрессия

**Курс «Анализ больших данных»**

---

### Как работать с этим блокнотом

1. Выполняйте ячейки **по порядку**: `Shift + Enter`.
2. Подойдёт Jupyter (Anaconda или python.org + pip) или Google Colab — см. инструкцию в информационном
   разделе курса. Нужны numpy, pandas, scipy, matplotlib; sklearn и statsmodels — по желанию (есть в Colab).
3. 📘 — разбор: запустите и посмотрите результат. ✏️ — задание: допишите код вместо `...` или `# ваш код`, затем запустите
   ячейку проверки с `assert` — если ошибок нет, решение верно.
   💡 — решение под треугольником, открывайте после честной попытки.
4. Перед сдачей: **Kernel → Restart & Run All**.

### Что будет к концу блокнота

Вы оцените модель цены отелей «КубаньТура» методом наименьших квадратов, проверите мультиколлинеарность,
закодируете качественные факторы, проведёте диагностику остатков, отберёте факторы и проверите модель
кросс-валидацией, а затем оцените модель цены бронирований «по частям» — как на больших данных. Каждое задание
проверяется ячейкой с `assert`.

In [ ]:
import json, os, sys, time, random
from datetime import datetime, timedelta, timezone
from pprint import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 140)
BLUE, ORANGE = "#34519E", "#CF7A2A"      # цвета серий на графиках
NAVY, GREY = "#1E2761", "#9AA3B5"
IN_COLAB = "google.colab" in sys.modules
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| Colab" if IN_COLAB else "| локально")

Данные — кейс **«КубаньТур»** (условные; первая ячейка работает ≈ 3 с). Запустите её.

In [ ]:
# Сквозной кейс курса «Анализ больших данных» — «КубаньТур»: условная платформа бронирования
# средств размещения Краснодарского края. Все отели, туристы и бронирования вымышлены; генерация детерминирована
# (seed 2026), у всех получаются одинаковые числа. Только стандартная библиотека.
#   HOTELS          — 240 средств размещения: город, тип, звёзды, номера, персонал, расстояние до моря, удобства,
#                     рейтинг, базовая цена номера в сутки (высокий сезон)
#   bookings(n)     — ЛЕНИВЫЙ генератор n бронирований (словари) за 01.01.2025–30.09.2026
#   BOOKINGS        — 60 000 бронирований (список словарей) — основной набор для pandas
#   booking_lines(n)— ленивый генератор строк CSV «id;дата;отель;город;канал;ночей;гостей;сумма;отмена»
#   RAW_BOOKINGS    — 20 000 строк «сырой» выгрузки с дефектами (см. RAW_DEFECTS) — для разведочного анализа
import math as _m
import random as _random
from datetime import date as _date, timedelta as _td

_SEED = 2026
_TITLE = "КубаньТур"
_R = _random.Random(_SEED)
_D0, _D1 = _date(2025, 1, 1), _date(2026, 9, 30)

_CITIES = [  # город, доля отелей, у моря?, эффект цены (лог), «горный»?
    ("Сочи", 0.28, True, 0.30, False), ("Анапа", 0.17, True, 0.00, False), ("Геленджик", 0.14, True, 0.15, False),
    ("Краснодар", 0.12, False, 0.05, False), ("Красная Поляна", 0.09, False, 0.35, True),
    ("Туапсе", 0.08, True, -0.10, False), ("Ейск", 0.06, True, -0.25, False), ("Абрау-Дюрсо", 0.06, True, 0.20, False)]
_TYPES = [  # тип, доля, звёзды (мин, макс), номера (мин, макс), персонал на номер, эффект цены (лог)
    ("отель", 0.40, 2, 5, 40, 400, 0.45, 0.10), ("гостевой дом", 0.25, 0, 2, 5, 25, 0.15, -0.15),
    ("апартаменты", 0.15, 0, 3, 20, 120, 0.12, 0.05), ("санаторий", 0.10, 3, 4, 100, 500, 0.80, 0.20),
    ("хостел", 0.10, 0, 0, 10, 40, 0.10, -0.80)]
_WORDS = ["Бриз", "Волна", "Магнолия", "Кипарис", "Лазурь", "Прибой", "Пальма", "Чайка", "Маяк", "Риф", "Парус",
          "Гранат", "Лаванда", "Олива", "Горизонт", "Альпика", "Эдельвейс", "Кубань", "Таврия", "Фрегат", "Аврора",
          "Ривьера", "Жемчуг", "Солнце", "Ласточка", "Каштан", "Платан", "Янтарь", "Нептун", "Медуза"]
_PREFIX = {"отель": "Отель", "гостевой дом": "Гостевой дом", "апартаменты": "Апартаменты",
           "санаторий": "Санаторий", "хостел": "Хостел"}


def _pick(r, items, weights):
    return r.choices(items, weights=weights, k=1)[0]


HOTELS = []
for _i in range(1, 241):
    _c = _pick(_R, _CITIES, [c[1] for c in _CITIES])
    _t = _pick(_R, _TYPES, [t[1] for t in _TYPES])
    _stars = _R.randint(_t[2], _t[3])
    _rooms = _R.randint(_t[4], _t[5])
    _staff = max(1, round(_rooms * _t[6] * _R.lognormvariate(0, 0.15)))
    if _c[2]:
        _dist = round(min(4.0, _R.lognormvariate(_m.log(0.6), 0.8)), 2)
    elif _c[4]:
        _dist = round(40 + _R.uniform(-3, 3), 1)
    else:
        _dist = round(_R.uniform(115, 150), 1)
    _pool = _R.random() < 0.10 + 0.15 * _stars
    _spa = _R.random() < (0.05 + 0.12 * _stars + (0.4 if _t[0] == "санаторий" else 0))
    _breakfast = _R.random() < 0.3 + 0.12 * _stars
    _parking = _R.random() < 0.55 + (0.2 if not _c[2] else 0)
    _area = round(16 + 4.2 * _stars + _R.gauss(0, 3) + (12 if _t[0] == "апартаменты" else 0), 1)
    _rating = round(min(9.9, max(5.0, 6.4 + 0.45 * _stars - 0.25 * _m.log(_dist + 0.1) * (1 if _c[2] else 0)
                                + 0.3 * _spa + _R.gauss(0, 0.45))), 1)
    _lp = (_m.log(1600) + 0.33 * _stars + 0.22 * _pool + 0.18 * _spa + 0.10 * _breakfast + 0.012 * (_area - 25)
           - (0.12 * _m.log(_dist + 0.1) if _c[2] else 0) + _c[3] + _t[7] + 0.08 * (_rating - 8) + _R.gauss(0, 0.12))
    HOTELS.append({"hotel_id": f"H{_i:03d}", "name": f"{_PREFIX[_t[0]]} «{_R.choice(_WORDS)}»", "city": _c[0],
                   "type": _t[0], "stars": _stars, "rooms": _rooms, "staff": _staff, "dist_sea_km": _dist,
                   "pool": _pool, "spa": _spa, "breakfast": _breakfast, "parking": _parking, "room_area": _area,
                   "rating": _rating, "price_base": int(round(_m.exp(_lp), -1))})
_HOTEL = {h["hotel_id"]: h for h in HOTELS}
_CITY = {c[0]: c for c in _CITIES}
_REGIONS = [("Москва", 0.24), ("Краснодарский край", 0.18), ("Московская обл.", 0.10), ("Санкт-Петербург", 0.09),
            ("Ростовская обл.", 0.08), ("Самарская обл.", 0.05), ("Свердловская обл.", 0.05),
            ("Республика Татарстан", 0.05), ("Воронежская обл.", 0.04), ("другие", 0.12)]
_CHANNELS = [("сайт", 0.25, 1.00), ("приложение", 0.30, 0.97), ("агрегатор", 0.35, 1.06), ("офис", 0.10, 1.03)]
_AGES = [("18–24", 0.12), ("25–34", 0.30), ("35–44", 0.27), ("45–54", 0.16), ("55+", 0.15)]


def _season(month: int, city: tuple) -> float:
    """Сезонный множитель цены и спроса: море — пик в июле–августе, горы — зимой и летом, Краснодар — ровно."""
    if city[4]:
        return [1.25, 1.30, 1.05, 0.70, 0.75, 0.90, 1.05, 1.10, 0.85, 0.70, 0.75, 1.20][month - 1]
    if not city[2]:
        return [0.85, 0.85, 0.95, 1.00, 1.05, 1.05, 1.00, 1.00, 1.05, 1.05, 0.95, 1.00][month - 1]
    return [0.45, 0.45, 0.50, 0.60, 0.80, 1.05, 1.35, 1.40, 1.00, 0.65, 0.50, 0.55][month - 1]


def bookings(n: int, seed: int = _SEED):
    """Ленивый генератор n бронирований (словари). Одинаковый seed — одинаковые данные."""
    r = _random.Random(seed + 1)
    span = (_D1 - _D0).days
    hotel_w = [h["rooms"] ** 0.7 * (1 + (h["rating"] - 7) / 3) for h in HOTELS]
    for k in range(1, n + 1):
        h = r.choices(HOTELS, weights=hotel_w, k=1)[0]
        c = _CITY[h["city"]]
        while True:                                             # дата заезда с учётом сезонного спроса
            d = _D0 + _td(days=r.randrange(span + 1))
            if r.random() < _season(d.month, c) / 1.45:
                break
        ch = _pick(r, _CHANNELS, [x[1] for x in _CHANNELS])
        lead = min(240, int(r.expovariate(1 / (35 if ch[0] != "офис" else 55))))
        nights = max(1, min(21, int(r.lognormvariate(_m.log(4.2 if c[2] else (3.2 if c[4] else 1.8)), 0.55))))
        guests = r.choices([1, 2, 3, 4], weights=[18, 50, 18, 14], k=1)[0]
        if h["type"] == "хостел":
            guests = 1
        wknd = 1.08 if d.weekday() >= 4 else 1.0
        disc = 0.92 if lead > 60 else 1.0
        price = h["price_base"] * _season(d.month, c) * wknd * disc * ch[2] * r.lognormvariate(0, 0.08)
        price *= 1 + 0.25 * (guests - 2) * (h["type"] in ("апартаменты", "санаторий"))
        price = int(round(max(500, price), -1))
        p_cancel = 0.06 + 0.07 * (ch[0] == "агрегатор") + 0.0006 * lead
        cancelled = r.random() < p_cancel
        score = None
        if not cancelled and r.random() < 0.35:
            score = max(1, min(10, round(h["rating"] + r.gauss(0, 1.1) - (0.4 if guests >= 3 else 0))))
        yield {"booking_id": f"B{k:07d}", "created": (d - _td(days=lead)).isoformat(), "checkin": d.isoformat(),
               "hotel_id": h["hotel_id"], "city": h["city"], "type": h["type"], "channel": ch[0],
               "region": _pick(r, [x[0] for x in _REGIONS], [x[1] for x in _REGIONS]),
               "age_group": _pick(r, [x[0] for x in _AGES], [x[1] for x in _AGES]),
               "lead_days": lead, "nights": nights, "guests": guests, "price_night": price,
               "amount": price * nights, "cancelled": cancelled, "review_score": score}


BOOKINGS = list(bookings(60_000))


def booking_lines(n: int, seed: int = _SEED):
    """Ленивый генератор строк журнала «id;дата_заезда;отель;город;канал;ночей;гостей;сумма;отмена» (без заголовка)."""
    for b in bookings(n, seed):
        yield (f"{b['booking_id']};{b['checkin']};{b['hotel_id']};{b['city']};{b['channel']};{b['nights']};"
               f"{b['guests']};{b['amount']};{int(b['cancelled'])}")


# «сырая» выгрузка: те же поля, но с типичными дефектами реальных данных
_RR = _random.Random(_SEED + 7)
RAW_BOOKINGS, RAW_DEFECTS = [], {"формат даты дд.мм.гггг": 0, "город с ошибкой написания": 0,
                                 "сумма строкой «… руб.»": 0, "пропуск ночей": 0, "пропуск канала": 0,
                                 "выброс цены (×100)": 0, "выброс ночей (365)": 0, "отрицательные гости": 0,
                                 "полный дубль": 0}
for _b in BOOKINGS[:20_000]:
    _row = {k: _b[k] for k in ("booking_id", "checkin", "hotel_id", "city", "channel", "nights", "guests",
                               "price_night", "cancelled")}
    _u = _RR.random()
    if _u < 0.06:
        _row["checkin"] = _date.fromisoformat(_row["checkin"]).strftime("%d.%m.%Y"); RAW_DEFECTS["формат даты дд.мм.гггг"] += 1
    elif _u < 0.10:
        _row["city"] = _RR.choice([_row["city"].lower(), _row["city"] + " ", "г. " + _row["city"]]); RAW_DEFECTS["город с ошибкой написания"] += 1
    elif _u < 0.13:
        _row["price_night"] = f"{_row['price_night']:,} руб.".replace(",", " "); RAW_DEFECTS["сумма строкой «… руб.»"] += 1
    elif _u < 0.155:
        _row["nights"] = None; RAW_DEFECTS["пропуск ночей"] += 1
    elif _u < 0.175:
        _row["channel"] = None; RAW_DEFECTS["пропуск канала"] += 1
    elif _u < 0.180:
        _row["price_night"] = _row["price_night"] * 100; RAW_DEFECTS["выброс цены (×100)"] += 1
    elif _u < 0.184:
        _row["nights"] = 365; RAW_DEFECTS["выброс ночей (365)"] += 1
    elif _u < 0.187:
        _row["guests"] = -_row["guests"]; RAW_DEFECTS["отрицательные гости"] += 1
    RAW_BOOKINGS.append(_row)
    if _RR.random() < 0.015:
        RAW_BOOKINGS.append(dict(_row)); RAW_DEFECTS["полный дубль"] += 1

print(f"КубаньТур: отелей {len(HOTELS)}, бронирований {len(BOOKINGS)}, строк сырой выгрузки {len(RAW_BOOKINGS)}")

## 1. МНК 📘

Нормальные уравнения, стандартные ошибки, t, p, доверительные интервалы, R² и F — своими руками на numpy.

In [ ]:
# МНК своими руками: коэффициенты, стандартные ошибки, t, p, ДИ, R², F
from scipy import stats

def ols(X, y, names):
    X, y = np.asarray(X, float), np.asarray(y, float)
    n, k = X.shape
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ beta
    rss, tss = e @ e, (y - y.mean()) @ (y - y.mean())
    s2 = rss / (n - k)
    se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    t = beta / se
    q = stats.t.ppf(0.975, n - k)
    r2 = 1 - rss / tss
    F = (r2 / (k - 1)) / ((1 - r2) / (n - k))
    tab = pd.DataFrame({"коэф.": beta, "ст. ош.": se, "t": t, "p": 2 * stats.t.sf(abs(t), n - k),
                        "ДИ от": beta - q * se, "ДИ до": beta + q * se}, index=names)
    return {"tab": tab, "beta": beta, "resid": e, "fit": X @ beta, "n": n, "k": k, "rss": rss, "s2": s2,
            "r2": r2, "r2_adj": 1 - (1 - r2) * (n - 1) / (n - k), "F": F, "F_p": stats.f.sf(F, k - 1, n - k)}

h = pd.DataFrame(HOTELS)
h["lp"] = np.log(h.price_base)
m0 = ols(np.column_stack([np.ones(len(h)), h.stars, h.pool]), h.lp, ["const", "stars", "pool"])
print(m0["tab"].round(3).to_string(), f"\nR² = {m0['r2']:.3f}")

## 2. Мультиколлинеарность 📘

Корреляционная матрица и VIF = 1 / (1 − R²ⱼ).

In [ ]:
# Корреляционная матрица и VIF
num = ["stars", "rooms", "staff", "room_area", "rating", "dist_sea_km"]

def vif(d, cols):
    out = {}
    for j in cols:
        others = [c for c in cols if c != j]
        Xj = np.column_stack([np.ones(len(d))] + [d[c].astype(float) for c in others])
        out[j] = 1 / (1 - ols(Xj, d[j].astype(float), ["const"] + others)["r2"])
    return pd.Series(out).round(1)

print(h[num + ["lp"]].corr().round(2)["lp"].to_dict())
vif(h, num)

## 3. Спецификация 📘

Фиктивные переменные с эталонной категорией; логарифм расстояния только у моря.

In [ ]:
# Спецификация: фиктивные переменные (эталон — отель в Краснодаре), взаимодействие «у моря × ln расстояния»
BASE = {"city": "Краснодар", "type": "отель"}

def design(d, cols, base=BASE):
    parts, names = [np.ones(len(d))], ["const"]
    for c in cols:
        if c in base:
            for v in sorted(d[c].unique()):
                if v != base[c]:
                    parts.append((d[c] == v).to_numpy(float)); names.append(f"{c}={v}")
        else:
            parts.append(d[c].to_numpy(float)); names.append(c)
    return np.column_stack(parts), names

h["ldist_sea"] = np.where(h.dist_sea_km < 10, np.log(h.dist_sea_km + 0.1), 0.0)
FULL = ["stars", "pool", "spa", "breakfast", "parking", "room_area", "rating", "rooms", "staff", "ldist_sea", "city", "type"]
X, names = design(h, FULL)
m1 = ols(X, h.lp, names)
print(f"R² = {m1['r2']:.4f}, скорр. R² = {m1['r2_adj']:.4f}, F = {m1['F']:.0f}")
m1["tab"].assign(**{"эффект, %": np.expm1(m1["tab"]["коэф."]) * 100}).round(3)

## 4. Диагностика остатков 📘

Нормальность, гетероскедастичность, влиятельные наблюдения.

In [ ]:
# Диагностика остатков: QQ, остатки против прогноза, Бройш–Паган, расстояние Кука
e, n, k = m1["resid"], *X.shape
lm = n * ols(X, e ** 2, names)["r2"]
lev = np.einsum("ij,jk,ik->i", X, np.linalg.inv(X.T @ X), X)
cook = e ** 2 / (k * m1["s2"]) * lev / (1 - lev) ** 2
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
stats.probplot(e, plot=ax[0]); ax[0].set_title("QQ остатков")
ax[1].scatter(m1["fit"], e, s=8, color=ORANGE); ax[1].axhline(0, color=GREY); ax[1].set_xlabel("прогноз"); ax[1].set_ylabel("остаток")
ax[2].stem(cook, markerfmt=" ", basefmt=" "); ax[2].axhline(4 / n, color=ORANGE, ls="--"); ax[2].set_title("расстояние Кука")
plt.tight_layout(); plt.show()
print(f"Шапиро p = {stats.shapiro(e).pvalue:.3f}; Бройш–Паган p = {stats.chi2.sf(lm, k - 1):.3f}; макс. D Кука = {cook.max():.3f}")

## 5. Корректировка модели 📘

Вложенные модели, информационные критерии, кросс-валидация.

In [ ]:
# Корректировка: F для вложенных моделей, AIC/BIC, кросс-валидация
REDUCED = ["stars", "pool", "spa", "breakfast", "room_area", "rating", "ldist_sea", "city", "type"]
X2, names2 = design(h, REDUCED)
m2 = ols(X2, h.lp, names2)
q = X.shape[1] - X2.shape[1]
F = ((m2["rss"] - m1["rss"]) / q) / (m1["rss"] / (n - X.shape[1]))
print(f"F = {F:.2f}, p = {stats.f.sf(F, q, n - X.shape[1]):.2f}")

def aic_bic(m):
    base = m["n"] * np.log(m["rss"] / m["n"])
    return round(float(base + 2 * m["k"]), 1), round(float(base + m["k"] * np.log(m["n"])), 1)

def cv_rmse(X, y, k=5, seed=1):
    y = np.asarray(y, float)
    idx = np.random.default_rng(seed).permutation(len(y))
    err = []
    for part in np.array_split(idx, k):
        tr = np.setdiff1d(idx, part)
        b_ = np.linalg.lstsq(X[tr], y[tr], rcond=None)[0]
        err.append((y[part] - X[part] @ b_) ** 2)
    return np.sqrt(np.concatenate(err).mean())

for name, m, Xm in [("полная", m1, X), ("сокращённая", m2, X2)]:
    print(name, "AIC, BIC =", aic_bic(m), "| CV RMSE =", round(cv_rmse(Xm, h.lp), 4))

## 6. Регрессия на больших данных 📘

Суммы XᵀX и Xᵀy по блокам — как MapReduce.

In [ ]:
# МНК «по частям» на 60 000 бронированиях: суммы XᵀX и Xᵀy по блокам
base_price = h.set_index("hotel_id").price_base
sea_city = set(h.loc[h.dist_sea_km < 10, "city"])
CH = ["приложение", "агрегатор", "офис"]

def block(rows):
    d = pd.DataFrame(rows)
    d = d[d.city.isin(sea_city) & d.type.isin(["отель", "гостевой дом"])]
    dt = pd.to_datetime(d.checkin)
    y = np.log(d.price_night / d.hotel_id.map(base_price)).to_numpy()
    Xb = np.column_stack([(dt.dt.month.to_numpy()[:, None] == np.arange(1, 13)).astype(float),
                          (dt.dt.dayofweek >= 4).to_numpy(float), (d.lead_days > 60).to_numpy(float)]
                         + [(d.channel == c).to_numpy(float) for c in CH])
    return Xb, y

XtX, Xty, N = np.zeros((17, 17)), np.zeros(17), 0
for i in range(0, len(BOOKINGS), 10_000):
    Xb, yb = block(BOOKINGS[i:i + 10_000])
    XtX += Xb.T @ Xb; Xty += Xb.T @ yb; N += len(yb)
beta = np.linalg.solve(XtX, Xty)
print(N, dict(zip(["выходные", "раннее"] + CH, np.round(np.exp(beta[12:]), 3))))
print("сезон:", np.round(np.exp(beta[:12]), 2).tolist())

## 7. Библиотеки 📘

statsmodels даёт те же оценки и готовую сводку.

In [ ]:
# То же в statsmodels (если установлен; в Colab есть)
try:
    import statsmodels.formula.api as smf
    sm_m = smf.ols("lp ~ stars + pool + spa + breakfast + room_area + rating + ldist_sea"
                   " + C(city, Treatment('Краснодар')) + C(type, Treatment('отель'))", data=h).fit(cov_type="HC1")
    print(sm_m.summary().tables[0])
    print("совпадает с нашей функцией:", np.allclose(sm_m.params.to_numpy()[[0]], m2["beta"][[0]]))
except ImportError:
    print("statsmodels не установлен — пропускаем (pip install statsmodels)")

---

# Задания ✏️

Пишите код вместо `...` / `# ваш код`, затем запускайте **ячейку проверки** под заданием.

## Задание 1. Модель и интерпретация

Эффекты в процентах и доверительный интервал.

In [ ]:
# Задание 1. Оцените модель ln цены ~ константа + stars + spa + room_area (функция ols). Найдите r2_t1 (R²),
# эффекты в процентах pct (словарь: фактор -> (e^β − 1)·100) и 95 % ДИ эффекта СПА в процентах spa_ci (кортеж).
# ваш код

In [ ]:
assert round(float(r2_t1), 4) == 0.9063
assert set(pct) == {"stars", "spa", "room_area"} and spa_ci[0] < pct["spa"] < spa_ci[1]
print("Задание 1: верно")

## Задание 2. VIF

Мультиколлинеарность и замена пары факторов отношением.

In [ ]:
# Задание 2. Посчитайте VIF для факторов stars, rating, room_area, rooms, staff, spa, pool (spa и pool как 0/1) —
# vif7 (Series). Список факторов с VIF > 5 — high. Затем замените rooms и staff одним фактором staff_per_room =
# staff / rooms и пересчитайте VIF — vif6. Как изменились VIF у пары «номера — персонал»?
# ваш код

In [ ]:
assert set(high) == {"stars", "rooms", "staff"}
assert vif6["rooms"] < 3 and vif6["staff_per_room"] < 3
print("Задание 2: верно")

## Задание 3. Эталонная категория

Смена эталона не меняет модель.

In [ ]:
# Задание 3. Оцените сокращённую модель (REDUCED) с другим эталоном типа — «хостел» (city — по-прежнему Краснодар):
# m_host. Найдите эффект «отель против хостела» в процентах hotel_vs_hostel. Проверьте, что разница коэффициентов
# «отель» и «гостевой дом» не зависит от выбора эталона: diff_new (в m_host) и diff_old (в m2, где эталон — отель).
# ваш код

In [ ]:
assert abs(m_host["r2"] - m2["r2"]) < 1e-10, "смена эталона не меняет модель"
assert abs(diff_new - diff_old) < 1e-8 and abs(c_new["type=отель"] + c_old["type=хостел"]) < 1e-8
assert 120 < hotel_vs_hostel < 200
print("Задание 3: верно")

## Задание 4. Остатки простой модели

Систематические ошибки спецификации.

In [ ]:
# Задание 4. Диагностика простой модели m0 (stars, pool): средний остаток по типам размещения resid_by_type (Series)
# и p-значение Бройша–Пагана bp_p0. Какие типы модель систематически переоценивает и недооценивает? Что это говорит
# о спецификации?
# ваш код

In [ ]:
assert resid_by_type.idxmin() == "хостел" and resid_by_type["хостел"] < -0.4
assert resid_by_type.idxmax() == "апартаменты"
assert 0 <= bp_p0 <= 1
print("Задание 4: верно")

## Задание 5. Устойчивые стандартные ошибки

Формула HC1.

In [ ]:
# Задание 5. Функция hc1(X, e) возвращает устойчивые стандартные ошибки HC1:
# sqrt(diag(inv(XᵀX) · Xᵀ diag(e²) X · inv(XᵀX)) · n/(n − k)). Посчитайте их для сокращённой модели m2 — se_hc (Series
# с индексом names2) и отношение ratio = se_hc / обычные ст. ошибки. В каких пределах оно меняется?
def hc1(X, e):
    ...

In [ ]:
Xt_ = np.column_stack([np.ones(5), [1, 2, 3, 4, 6.]]); et_ = np.array([.1, -.2, .1, .1, -.1])
manual = np.sqrt(np.diag(np.linalg.inv(Xt_.T @ Xt_) @ (Xt_.T @ np.diag(et_ ** 2) @ Xt_) @ np.linalg.inv(Xt_.T @ Xt_)) * 5 / 3)
assert np.allclose(hc1(Xt_, et_), manual)
assert list(se_hc.index) == names2 and 0.5 < ratio.min() and ratio.max() < 2
print("Задание 5: верно")

## Задание 6. Отбор факторов

Пошаговое исключение по p-значению.

In [ ]:
# Задание 6. Пошаговый отбор «назад»: начните с FULL; пока среди КОЛИЧЕСТВЕННЫХ и бинарных факторов есть p > 0,05,
# исключайте один фактор с наибольшим p и переоценивайте модель (city и type не исключать). Порядок исключения —
# dropped (список), итоговые факторы — selected.
# ваш код

In [ ]:
assert dropped == ["parking", "staff", "rooms"]
assert selected == REDUCED
print("Задание 6: верно")

## Задание 7. Форма связи

Сравнение спецификаций кросс-валидацией.

In [ ]:
# Задание 7. Как лучше учесть расстояние до моря? Сравните кросс-валидацией (cv_rmse) три спецификации сокращённой
# модели, где вместо ldist_sea стоит: (а) dist_sea_km в километрах; (б) ln(dist_sea_km + 0,1) у всех объектов;
# (в) ldist_sea. Результат — словарь cv (ключи "км", "ln все", "ln у моря").
# ваш код

In [ ]:
assert cv["км"] > 1.1 * max(cv["ln все"], cv["ln у моря"])
assert abs(cv["ln все"] - cv["ln у моря"]) < 0.005
print("Задание 7: верно")

## Задание 8 ★. Мини-исследование: модель цены бронирования

МНК по блокам, сезонность групп городов.

In [ ]:
# Задание 8 ★ (мини-исследование). Модель цены бронирования для типов «отель», «апартаменты», «санаторий» во ВСЕХ
# городах: y = ln(price_night / price_base); факторы — 36 фиктивных «месяц × группа города» (у моря / горы — Красная
# Поляна / Краснодар; без константы), выходные, раннее бронирование (> 60 дней), каналы (эталон — сайт) и число
# гостей 1, 3, 4 у апартаментов и санаториев (эталон — 2 гостя). Оцените МНК по блокам по 5000 строк (XtX8, Xty8),
# проверьте совпадение с lstsq по всем данным (same), таблица множителей mult8 (e^β) для выходных, раннего
# бронирования, каналов и числа гостей; сезонные множители трёх групп — season8 (DataFrame 12 × 3). Вывод 5–7
# предложений: какие эффекты, как различается сезонность групп.
# ваш код

In [ ]:
assert same and XtX8.shape == (44, 44)
assert abs(mult8["выходные"] - 1.08) < 0.01 and abs(mult8["агрегатор"] - 1.06) < 0.01
assert mult8["4 гостя"] > mult8["3 гостя"] > 1 > mult8["1 гость"]
assert season8.loc[8, "у моря"] > 1.3 and season8.loc[2, "горы"] > season8.loc[2, "у моря"]
print("Задание 8: таблицы построены; вывод проверит преподаватель")

---

## Что вы теперь умеете

| Задача | Инструмент |
|---|---|
| МНК | `np.linalg.lstsq`, нормальные уравнения, ст. ошибки, t, p, ДИ |
| мультиколлинеарность | `corr`, VIF |
| спецификация | фиктивные переменные, эталон, логарифмы, взаимодействия |
| диагностика | QQ, Бройш–Паган, рычаг, расстояние Кука, HC1 |
| отбор | F для вложенных моделей, AIC/BIC, кросс-валидация |
| большие данные | суммы XᵀX и Xᵀy по блокам |

**Дальше — тема 5 «Многомерные методы».** Главные компоненты, факторный и кластерный анализ,
дискриминантный анализ.